In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# APIs
from fredapi import Fred

# plotting
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# import py-files
import states

# 1. Real GDP Across US States

## Question 1.1
### Question 1.1.1

We start by acessing data from the FRED through their API. To access data from FRED you need to:

1. Create a FRED account [here](https://fredaccount.stlouisfed.org/login/secure/).
2. Request an API key [here](https://fredaccount.stlouisfed.org/apikeys).
3. Save in otherwise empty file `fredapi.txt` somewhere on your computer.

For this code to run, you must change the path to the location of the `fredapi.txt` on your computer.

In [ ]:
# TODO: Change path to your own path to fredapi.txt file
with open('/Users/simonskodeberg/Desktop/Københavns Universitet/Programming files/fredapi.txt', 'r') as f: fred_api_key = f.read()
fred = Fred(api_key=fred_api_key)

In [ ]:
# Makes download function to not repeat the download logic twice.
# However, it makes 100 separate requests to the FRED API. One for
# each state and each variable. This makes the processing relavetive slow.

def download_fred_panel(codes):
    """ download a dict of {fred_code: state} from FRED and collect the series
    in a data frame with years in the index and states in the columns """

    data = {}
    for code, state in codes.items():
        var = fred.get_series(code)
        data[state] = var.resample('YS').mean() # resample to annual frequency (YS = year start)

    # Convert to data frame, set index to year, and sort columns by state name
    df = pd.DataFrame(data)
    df.index = df.index.year
    df = df.rename_axis('year').sort_index(axis=1)

    return df

# a. Builds the two dictionaries for real GDP (XXRGSP) and population (XXPOP), for all 50 states
rgsp_codes = {f'{state}RGSP': state for state in states.STATES}
pop_codes = {f'{state}POP': state for state in states.STATES}

# b. Download and collect in two data frames
RGSP = download_fred_panel(rgsp_codes)
POP = download_fred_panel(pop_codes)

RGSP.head()
POP.head()

Thus, we have used the FRED API to downlaod real GDP and population data for the for the individual US states, set the years as the index and the states as the columns.

### Question 1.1.2

The two two series do not cover the same years. We compare the two data frames to exmine which years they have in common. We only keep the years where the states both have a real GDP and population observation.

In [ ]:
# a. Keep only the years where both RGSP and POP have data
common_years = RGSP.index.intersection(POP.index)
RGSP = RGSP.loc[common_years]
POP = POP.loc[common_years]

# b. Drop any state with a missing observation in those years
valid_states = RGSP.columns[RGSP.notna().all() & POP.notna().all()] # Returns true only if every year in the column (for this state) is non-missing.
dropped_states = list(RGSP.columns.difference(valid_states))

RGSP = RGSP[valid_states]
POP = POP[valid_states]

print(f'The two data frames have {len(common_years)} common years which are {common_years.min()} to {common_years.max()}')
print(f'Thus, we have kept {len(common_years)} years and {len(valid_states)} states.')

if len(valid_states) < 50:
    print(f'The states that were dropped are: {dropped_states}')

### Question 1.1.3

We now compute the real GDP per person in dollars. Since, the real GDP and the population do not have the same unit of mesures, we multiply the ratio by 1,000 to get the from from millions / thousands = thousands of dollars per person to dollars per person.

In [ ]:
# Compute the per-capita real GDP and store in a new data frame y

y = RGSP / POP * 1000 # Multiply by 1000 to take units into account
y = y.rename_axis('year').rename_axis('state', axis=1)

y.head()

Thus, we have computed a date frame with the real GDP per person in dollars for each 50 states in the years 1997-2025

### Question 1.1.4

From question 1.1.2 we can conclude that $t_{first} = 1997$ and $t_{last} = 2025$. We now summary table that show the states with the highest and the lowest GDP per person in 1997 and 2025, the ratio between the highest and the lowest in both years, and the average across states in both years.

In [ ]:
# a. First and last year in the sample
t_first, t_last = y.index.min(), y.index.max()

# b. For each of the two years, find the highest/lowest state, the ratio, and the average
summary = pd.DataFrame(index=[t_first, t_last],
                        columns=['highest_state', 'highest_y', 'lowest_state', 'lowest_y', 'ratio', 'average'])

for t in [t_first, t_last]:
    highest_state = y.loc[t].idxmax()
    lowest_state = y.loc[t].idxmin()
    highest_y = y.loc[t, highest_state]
    lowest_y = y.loc[t, lowest_state]

    summary.loc[t] = [states.NAMES[highest_state], highest_y, states.NAMES[lowest_state], lowest_y, highest_y/lowest_y, y.loc[t].mean()]

summary = summary.rename_axis('year')
summary.style.format({'highest_y': '${:,.0f}', 'lowest_y': '${:,.0f}', 'ratio': '{:.2f}', 'average': '${:,.0f}'})

In 1997, Delaware had the highest real GDP per person ($69,749) and Idaho the lowest ($32,063), a ratio of 2.18. By 2025, New York topped the ranking ($94,702) and Mississippi was lowest ($42,434), a ratio of 2.23. The cross-state average has increased over the period, from $44,593 to $64,674, while the highest-to-lowest ratio changed only marginally. This suggests that although states have grown substantially richer on average but the relative dispersion in GDP per person across states has remained roughly the same.

### Question 1.1.5

We make a figure with one panel that shows the GDP per person across the time periode for two historical swing states (Pennsylvania, Wisconsin), Republican states (Oklahoma, West Virginia) and Democratic states (California, Illinois). We have use the results from the presidential election from [usafact.org](https://usafacts.org/articles/how-red-or-blue-is-your-state/) to select the relevant states. The second panel shows the GDP per person divided by the average across states in the same year, for all states.

In [ ]:
# a. Groups of states, colored by political lean, with a distinct linestyle per state within a group
groups = {
    'PA': ('Swing', 'tab:purple', '-'),
    'WI': ('Swing', 'tab:purple', '--'),
    'OK': ('Republican', 'tab:red', '-'),
    'WV': ('Republican', 'tab:red', '--'),
    'CA': ('Democratic', 'tab:blue', '-'),
    'IL': ('Democratic', 'tab:blue', '--'),
}

# b. Average across states, and GDP per person relative to that average
average = y.mean(axis=1) # axis = 1 means "across columns" i.e. across states
relative = y.div(average, axis=0) # axis = 0 means "across rows" i.e. for each year

# c. Shared x-axis ticks: 1997, then 2000, then every 5 years
xticks_time = [1997] + list(range(2000, y.index.max() + 1, 5))

# d. Figure with two panels
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel i: selected states, plus the average across all states
ax = axes[0]
for state, (label, color, ls) in groups.items():
    ax.plot(y.index, y[state], color=color, linestyle=ls, label=f'{states.NAMES[state]} ({label})')
ax.plot(y.index, average, color='black', linewidth=2.5, label='Average, all states')
ax.set_title('Real GDP per person, selected states')
ax.set_xlabel('Year')
ax.set_ylabel('Dollars per person')
ax.set_xticks(xticks_time)
ax.set_yticks(np.arange(20_000, 100_001, 10_000)) # 100_001 is used to include 100_000 in the ticks
ax.legend(fontsize=10, ncol=2) # The legend holds for both panels

# Panel ii: GDP per person relative to the average, for all states
ax = axes[1]
for state in y.columns:
    if state not in groups:
        ax.plot(y.index, relative[state], color='lightgray', linewidth=0.8, zorder=0.5)
for state, (label, color, ls) in groups.items():
    ax.plot(y.index, relative[state], color=color, linestyle=ls, linewidth=1.5, zorder=2)
ax.axhline(1, color='black', linewidth=1, zorder=1)
ax.set_title('Real GDP per person relative to the average, all states')
ax.set_xlabel('Year')
ax.set_ylabel('Ratio to average')
ax.set_xticks(xticks_time)
ax.set_yticks(np.arange(0.6, 1.81, 0.2))

fig.tight_layout()

In the left panel, the Democratic states consistently lie above the national average, while the Republican states consistently lie below it, with a gap of a similar order of magnitude throughout the sample. The swing states track close to the average. To clarify, this is not a causal relationship, and might not even reflect a correlation. It is only an expression of selection. Furthermore, we see that real GDP takes a dip in all six states around COVID-19.

In the right panel, dividing each state's GDP per person by the cross-state average in the same year expresses how rich or poor a state is relative to the rest of the country in that year, rather than in absolute dollars. The 50 states are spread widely across this ratio, from roughly 0.6 to 1.8 times the average, showing that the gap between the richest and poorest states is large and persistent rather than closing over time.

## Question 1.2
### Question 1.2.1, 1.2.2, and 1.2.3

We compute the standard deviation across states of log GDP per person $(\log y_{i,t})$ each year, and plot it against the year.

In [ ]:
# a. Standard deviation across states of log y_it, for each year
std_log_y = np.log(y).std(axis=1) # axis = 1 means across columns (states) for each row (year).

# b. Plot against the year
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(std_log_y.index, std_log_y.values, color='black')
ax.set_title('Cross-state standard deviation of log real GDP per person')
ax.set_xlabel('Year')
ax.set_ylabel(r'Std. dev. of $\log y_{i,t}$')
ax.set_xticks(xticks_time)

fig.tight_layout()

The standard deviation of $\log y_{i,t}$ across states in a given year describes how spread out the states' income levels are that year. On this scale a higher value means the richest and poorest states are proportionally further apart and a lower value means the states are more similar to each other.

The plot shows the dispersion falling around the dot.com-crisis from 2001 to 2005, then rising sharply through the 2008–09 financial crisis to a peak around 2012–2015, before gradually declining again toward 2024. There is no single, clear trend over the full sample but instead moves with the business cycle.

We find the standard deviation of $\log y_{i,t}$ in 1997 and in 2025, and in which year it is highest and in which year it is lowest.

In [ ]:
# a. Standard deviation of log y_it in the first and last year
std_first = std_log_y.loc[t_first]
std_last = std_log_y.loc[t_last]

# b. Year with the highest and the lowest standard deviation
year_max = std_log_y.idxmax()
year_min = std_log_y.idxmin()

print(f'Std. dev. of log y in {t_first}:    {std_first:.3f}')
print(f'Std. dev. of log y in {t_last}:    {std_last:.3f}')
print('')
print(f'Highest std. dev.:  {std_log_y.max():.3f} in {year_max}')
print(f'Lowest std. dev.:   {std_log_y.min():.3f} in {year_min}')

## Question 1.3
### Question 1.3.1

For each state, we compute the average annual growth rate $g_i$ between 1997 and 2025, and make a scatter plot of $g_i$ against $\log y_{i,1997}$, to see whether initially poorer states grew faster than initially richer states.

In [ ]:
# a. Average annual growth rate for each state between t_first and t_last
T = t_last - t_first
g = (np.log(y.loc[t_last]) - np.log(y.loc[t_first])) / T

# b. Log GDP per person in t_first, for each state
log_y_first = np.log(y.loc[t_first])

# c. Scatter plot of g_i against log y_i,t_first
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(log_y_first, g, color='black')
ax.set_xlabel(fr'$\log y_{{i,{t_first}}}$')
ax.set_ylabel('$g_i$')
ax.set_title(fr'Average annual growth rate ($g_i$) vs. initial log GDP per person ($y_{{i,{t_first}}}$)')

fig.tight_layout()

$g_i$ is the average annual growth rate because it takes the total percent change in $y_i$ over the whole period, $\log y_{i,t_{last}} - \log y_{i,t_{first}}$, and spreads it evenly across the $T$ years. Thus, the average growth rate should be interpreted such that e.g. $g_i = 0.01$ is approximately a 1 percent growth rate per year on average. Even though actual growth in any single year may have been higher or lower, $g_i$ summarizes the whole path as if the state had grown at one constant rate throughout.

Even before fitting a linear regression, the scatter plot already suggests a negative relationship. Poorer states in 1997 tend to sit higher on average than richer states, though the relationship is noisy and it is correlation, not a causal relationship.

### Question 1.3.2, 1.3.3, 1.3.4, and 1.3.5

We now fit a straight line $g_i = a + b \log y_{i,t_{first}}$ through the scatter points using `np.polyfit`, and draw it in the same figure. We also label the five states with the highest $g_i$ and the five with the lowest $g_i$, and use the slope $b$ to compute the speed of convergence $\lambda$ and the half-life $h$.

In [ ]:
# a. Linear regression: g_i = a + b * log y_i,t_first
b, a = np.polyfit(log_y_first, g, 1)

# b. Correlation between g_i and log y_i,t_first
corr = np.corrcoef(log_y_first, g)[0, 1] # Returns a 2x2 corr-matrix and [0, 1] is the first row, second column, which is the correlation between the two variables.

# c. Speed of convergence and half-life
Lambda = -np.log(1 + b * T) / T
h = np.log(2) / Lambda

# d. Five states with the highest and the five with the lowest g_i
top5 = g.sort_values(ascending=False).index[:5] # ascending=False highest to lowest
bottom5 = g.sort_values(ascending=True).index[:5] # ascending=True lowest to highest
labeled_states = list(top5) + list(bottom5)
other_states = log_y_first.index.difference(labeled_states)

# e. Scatter plot with the fitted line, colored top-5/bottom-5 g_i states, and labels
fig, ax = plt.subplots(figsize=(9, 7))

# e.i. Makes three scatter plots to impliment different colors
ax.scatter(log_y_first[other_states], g[other_states], color='black', label='Other states')
ax.scatter(log_y_first[top5], g[top5], color='tab:green', label='Highest $g_i$ (top 5)')
ax.scatter(log_y_first[bottom5], g[bottom5], color='tab:orange', label='Lowest $g_i$ (bottom 5)')

# e.ii. Creates a fitted line through the endpionts
x_line = np.array([log_y_first.min(), log_y_first.max()]) # Linear line only needs two points to be defined
ax.plot(x_line, a + b * x_line, color='tab:red', linewidth=2,
        label=fr'Fitted line: $g_i = {a:.3f} {b:.3f} \cdot \log y_{{i,{t_first}}}$')

# e.iii. Add names to the top-5 and bottom-5 states
for state in labeled_states:
    ax.annotate(states.NAMES[state], (log_y_first[state], g[state]), textcoords='offset points', xytext=(-25, 5), fontsize=10)

ax.set_xlabel(fr'$\log y_{{i,{t_first}}}$')
ax.set_ylabel('$g_i$')
ax.set_title(fr'Average annual growth rate ($g_i$) vs. initial log GDP per person ($y_{{i,{t_first}}}$)')
ax.legend(fontsize=10)

fig.tight_layout()

# f. Report a, b, the correlation, lambda, and h
print(f'a = {a:.4f}')
print(f'b = {b:.4f}')
print(f'Correlation between g_i and log y_i,{t_first}: {corr:.3f}')
print('')
print(f'Speed of convergence, Lambda = {Lambda:.4f}')
print(f'Half-life, h = {h:.2f} years')

The fitted line has intercept $a = 0.118$ and slope $b = -0.010$, with a correlation of -0.39 between $g_i$ and $\log y_{i,1997}$. The negative relationship states that started poorer in 1997 tended to grow somewhat faster afterward. 

The points do not follow the line closely. For a simple linear regression the correlation coefficient maps strictly into the $R^2$-coeficient following $R^2 = corr^2$. Thus, the correlation of -0.39 means only about 15.2 percent of the variation in growth rates is explained by the initial income level.

The annual speed of convergence is $\lambda \approx 0.012$. This implies that it would take roughly 60.5 years for half of an initial income gap between two states to disappear at this rate.

## Question 1.4
### Question 1.4.1

We use `REGION` from `states.py` together with a `groupby` to compute, for each year, the average across the states in each of the four Census regions of $y_{i,t}$ divided by the average across all states in that year. We also report how many states there are in each region.

In [ ]:
# a. Map each state to its region
region_map = pd.Series(states.REGION, name='region').reindex(relative.columns) # Keys (state codes) become index and values (region names) become the data

# b. Number of states in each region
region_counts = region_map.value_counts().rename('states')
for region, count in region_counts.items():
    print(f'There are {count} states in the {region} region.')

# c. For each year, average across the states in each region of y_it / average across all states
regional_relative = relative.T.groupby(region_map).mean().T # transpose the date frame since groupby() operates on rows
regional_relative = regional_relative.rename_axis(columns='region')

regional_relative.head()

### Question 1.4.2

We plot the four regional series against the years in one figure from 1997 to 2025.

In [ ]:
# a. Plot the four regional series against the year
fig, ax = plt.subplots(figsize=(9, 7))
for region in regional_relative.columns:
    ax.plot(regional_relative.index, regional_relative[region], label=region)
ax.axhline(1, color='black', linewidth=1)
ax.set_xlabel('Year')
ax.set_ylabel('Regional average of $y_{i,t}$ / average, all states')
ax.set_title('Regional GDP per person relative to the national average')
ax.set_xticks(xticks_time)
ax.legend()

fig.tight_layout()

The Northeast stays richest throughout the time periode and the South poorest. The Northeast and the West stay at a roughly constant level throughout the periode. The South decreases through 2017 before their trend stay constant untill 2025. The Midwest trends up from untill 2014 before decreasing the rest of the time periode.

### Question 1.4.3

We report the value of each series in 1997 and 2025, along with the change between the two years, to see which region moved the most relative to the national average.

In [ ]:
# a. Value of each series in t_first and t_last, and the change between the two years
regional_summary = pd.DataFrame({ # Creates 4 rows (regions), 2 columns (1997, 2025)
    t_first: regional_relative.loc[t_first],
    t_last: regional_relative.loc[t_last],
})
regional_summary['change'] = regional_summary[t_last] - regional_summary[t_first]
regional_summary = regional_summary.reindex(regional_summary['change'].abs().sort_values(ascending=False).index) # Sorts by biggest mover

display(regional_summary.style.format('{:.3f}'))

# b. Which region moved the most, and in which direction
biggest_mover = regional_summary['change'].abs().idxmax()
direction = 'increased' if regional_summary.loc[biggest_mover, 'change'] > 0 else 'decreased'
print(f'{biggest_mover} moved the most. They {direction} by {regional_summary.loc[biggest_mover, "change"]:+.3f}.')

The pattern naturally follows the trends from the figure. The Northeast is the richest region relative to the national average throughout the sample around 1.10, followed by the West and the Midwest hovering close to 1.00, while the South consistently the poorest region 0.93. The South moved the most between 1997 and 2025, falling from 0.951 to 0.905. The Midwest moved the most in the opposite direction, rising from 0.968 to 1.006, crossing above the national average around 2011. The West and Northeast changed comparatively little, staying roughly where they started relative to the average.